# Coronary artery query

In this example, you will learn how to generate text from multiple images using the supported models: `Qwen2-VL`, `Pixtral` and `llava-interleaved`.

Multi-image generation allows you to pass a list of images to the model and generate text conditioned on all the images.


pip install -U mlx-vlm

In [ ]:
# Add this as the first code cell
import sys
import os

# Add the project root directory to Python path
project_root = "/Users/billb/github/mlx-vlm-Adjustment"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Verify the path is added
print(f"Added to Python path: {project_root}")

In [ ]:
from mlx_vlm import load, apply_chat_template, generate
from mlx_vlm.utils import load_image
from mlx_vlm.utils import process_image

In [7]:
# images = ["/Users/billb/Desktop/agram-70-50.png"]
# images = ["/Users/billb/Desktop/RCA-15-fps.png"]
# images = ["/Users/billb/Desktop/dcjcfjxoi919yd6c9iow68n7y.png"]
# images = ["/Users/billb/Desktop/dcjcfjxoi919yd6c9iow68n7y.model.png"]
images = ["/Users/billb/Desktop/No Name/Unnamed__0/Left_Coronary_15_fps_AWI_Correct_1/IM-0001-0054.jpg"] # angles -24 -19
# images = ["/Users/billb/Desktop/Passes/RCA-24-15.jpg"] # angles -24 -19
# images = ["/Users/billb/Desktop/Left Coronary 15 fps.jpg"]

# Convert images to grayscale
from PIL import Image
import numpy as np

grayscale_images = []
for img_path in images:
    # Open image and convert to grayscale
    img = Image.open(img_path).convert('L')
    grayscale_images.append(img_path)
    # Save grayscale version back to same path
    img.save(img_path)

images = grayscale_images

messages = [
    {"role": "user", "content": "This is a coronary angiogram image. The DICOM header reports PositionerPrimaryAngle=-24 and PositionerSecondaryAngle=-25. Here are the projection angle rules: PositionerPrimaryAngle < 0. -> Right, PositionerPrimaryAngle > 0. -> Left, PositionerSecondaryAngle < 0. -> Caudal, PositionerSecondaryAngle > 0. -> Cranial.  Please analyze the image and DICOM header data and tell me: 1. List the one or more coronary arteries being imaged and name each <coronary artery> ? 2. From what <projection view> is this image taken? Precede your response with the reasons for your analysis."}
]

messages = [
    # {"role": "user", "content": "This is a coronary angiogram image. The DICOM header reports PositionerPrimaryAngle=30.0 and PositionerSecondaryAngle=10.0. Here are the projection angle rules: PositionerPrimaryAngle < 0. -> Right, PositionerPrimaryAngle > 0. -> Left, PositionerSecondaryAngle < 0. -> Caudal, PositionerSecondaryAngle > 0. -> Cranial.  Please analyze the image and DICOM header data and respond in JSON format: 1. List the one or more coronary arteries being imaged and name each <coronary artery> 2. From what <projection view> is this image taken? Precede your response by placing in a JSON <comment> tag the reasons for your analysis."}
]
# messages = [
#     {"role": "user", "content": "This is a human coronary artery angiogram. Name the coronary artery or arteries being imaged? Does this image show a coronary artery stenosis? If so, state the stenosis location with bounding box coordinates. Be concise. Start by giving your reasoning."}
# ]

In [ ]:
# Display the first image
from PIL import Image
import matplotlib.pyplot as plt

# Load and display the image
img = Image.open(images[0])#.convert('L')  # Convert to grayscale mode
plt.figure(figsize=(10, 10))
plt.imshow(img,cmap='gray')
plt.axis('off')
plt.show()


## Qwen2-VL

In [ ]:
# Load model and processor
qwen_vl_model, qwen_vl_processor = load("mlx-community/Qwen2-VL-7B-Instruct-8bit")
# qwen_vl_model, qwen_vl_processor = load("mlx-community/Qwen2-VL-7B-Instruct-bf16")

qwen_vl_config = qwen_vl_model.config

In [ ]:
prompt = apply_chat_template(qwen_vl_processor, qwen_vl_config, messages, num_images=len(images))

In [ ]:
qwen_vl_output = generate(
    qwen_vl_model,
    qwen_vl_processor,
    images,
    prompt,
    max_tokens=1000,
    temperature=0.0, # 0.0 is deterministic
    verbose=True
)

## Pixtral

In [ ]:
# Load model and processor
pixtral_model, pixtral_processor = load("mlx-community/pixtral-12b-4bit")
# pixtral_model, pixtral_processor = load("mlx-community/pixtral-12b-8bit")
# pixtral_model, pixtral_processor = load("mlx-community/pixtral-12b-bf16")
pixtral_config = pixtral_model.config

In [ ]:

prompt = apply_chat_template(pixtral_processor, pixtral_config, messages, num_images=len(images))

In [11]:
# Pixtral requires images to be resized to the same shape in multi-image generation
resized_images = [process_image(load_image(image), (512, 512), None) for image in images]

In [ ]:
pixtral_output = generate(
    pixtral_model,
    pixtral_processor,
    resized_images,
    prompt,
    max_tokens=1000,
    temperature=0.0, # 0.0 is deterministic
    verbose=True
)

## Molmo

In [ ]:
# Load model and processor
model, processor = load("mlx-community/Molmo-7B-D-0924-4bit")
config = model.config

In [ ]:
prompt = apply_chat_template(processor, config, messages)

In [ ]:
molmo_output = generate(
    model,
    processor,
    images,
    prompt,
    max_tokens=5000,
    temperature=0.0,
)

## Llava-Interleaved

In [ ]:
# Load model and processor
llava_model, llava_processor = load("mlx-community/llava-interleave-qwen-0.5b-bf16")
llava_config = llava_model.config

In [12]:
prompt = apply_chat_template(llava_processor, llava_config, messages, num_images=len(images))

In [ ]:
llava_output = generate(
    llava_model,
    llava_processor,
    images,
    prompt,
    max_tokens=1000,
    temperature=0.0,
    verbose=True
)